# PA3 Qwen2.5 raw-only ? one offline Run All

Preparation candidate. GPU technical execution, metadata, TRAIN-dev and TRAIN-holdout each require a separate Research Lead decision and audited authority implementation. This version deliberately stops at AUTHORITY; no setting, receipt or hash unlocks it. TEST stays locked. See `notes/pa3_qwen_runbook.md`.

After the authority implementation and exact release are approved, attach the reviewed source bundle, existing pinned Qwen model prep and runtime prep plus its independently checked inventory. Select T4, Internet OFF. Fill the configuration once; Run All launches A?G in one child. No install/download, retry or fallback. A technical-only run uses four generated drawings; it does not evaluate hazard accuracy.


In [ ]:
# All attachment paths are relative to Kaggle's input directory.
# Dataset/cohort attachments require separate rights; images are never copied.
SETTINGS = {
    "mode": "technical",
    "run_id": "pa3-qwen-technical-first",
    "source_commit": "",
    "source_bundle_input": "",
    "source_bundle_sha256": "",
    "model_input": "",
    "runtime_input": "",
    "runtime_inventory_input": "",
    "runtime_inventory_sha256": "",
    "runtime_site_packages": "",
    "cohort_plan_input": "",
    "cohort_plan_sha256": "",
    "dataset_input": "",
}


In [ ]:
# A: verify trusted committed source before importing attached Python.
# B-G: launcher verifies prep, exact runtime, T1, qualification, cohort,
# authorized TRAIN inference, raw seal, CPU audit and descriptive evaluation.
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import subprocess
import sys
import tarfile

work = Path.cwd()
inputs = work.parent / "input"
run_id = SETTINGS.get("run_id", "")
if not re.fullmatch(r"pa3-qwen-[a-z0-9][a-z0-9_-]{0,63}", run_id):
    raise ValueError("SAFE_UNIQUE_RUN_ID_REQUIRED")
evidence = work / "data/processed/pa3_bootstrap" / run_id
evidence.mkdir(parents=True, exist_ok=False)
stage = "A_SOURCE_BUNDLE"

def write(name, value):
    payload = (json.dumps(value, sort_keys=True, allow_nan=False) + "\n").encode()
    with (evidence / name).open("xb") as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

try:
    sha = SETTINGS["source_commit"]
    if not re.fullmatch(r"[0-9a-f]{40}", sha):
        raise ValueError("REVIEWED_SOURCE_COMMIT_REQUIRED")
    rel = SETTINGS["source_bundle_input"]
    p = PurePosixPath(rel)
    if (not rel or p.is_absolute() or ".." in p.parts or "\\" in rel or ":" in rel):
        raise ValueError("INPUT_RELATIVE_BUNDLE_REQUIRED")
    bundle = inputs / p
    if not bundle.resolve().is_relative_to(inputs.resolve()):
        raise ValueError("BUNDLE_ESCAPES_INPUT")
    if digest(bundle) != SETTINGS["source_bundle_sha256"]:
        raise ValueError("TRUSTED_SOURCE_BUNDLE_HASH_MISMATCH")
    repo = work / "data/processed/pa3_source" / run_id
    if repo.exists():
        raise FileExistsError("SOURCE_EXISTS_NO_RETRY")
    subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "--no-checkout", str(bundle), str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "-c", "core.autocrlf=false", "checkout", "--detach", sha], check=True)
    head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    if head != sha:
        raise ValueError("SOURCE_COMMIT_MISMATCH")
    write("source.json", {"source_commit": sha, "bundle_sha256": SETTINGS["source_bundle_sha256"]})
    if any(k == "safeshift" or k.startswith("safeshift.") for k in sys.modules):
        raise ValueError("FRESH_NOTEBOOK_KERNEL_REQUIRED")
    sys.path.insert(0, str(repo))
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
                       "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_DISABLE_XET": "1",
                       "HF_HUB_ENABLE_HF_TRANSFER": "0", "CUDA_VISIBLE_DEVICES": "0"})
    from safeshift.pa3.native.kaggle import launch
    from safeshift.runners.internvl3_snapshot import network_denied
    stage = "B_TO_G_AUTHORIZED_WORKFLOW"
    with network_denied():
        output = launch(repo, inputs, SETTINGS)
    write("finish.json", {"status": "WORKFLOW_COMPLETE", "artifacts": output.relative_to(work).as_posix()})
    print("PA3 artifacts:", output.relative_to(work))
except BaseException as exc:
    write("failure.json", {"stage": stage, "exception_type": type(exc).__name__,
                           "retry_permitted": False, "runtime_pass_claim": False})
    raise
finally:
    write("checksums.json", {p.name: {"sha256": digest(p), "size_bytes": p.stat().st_size}
                              for p in sorted(evidence.iterdir()) if p.is_file()})
    archive = evidence.with_suffix(".tar.gz")
    with tarfile.open(archive, "x:gz") as stream:
        stream.add(evidence, arcname=evidence.name)
    with archive.with_suffix(".sha256.json").open("x", encoding="utf-8") as stream:
        json.dump({"archive": archive.name, "sha256": digest(archive)}, stream)
    print("Bootstrap evidence:", archive.relative_to(work))
